# Project 14 — BROKEN notebook (HMM pitfalls)

Seeded bugs centred on (1) state-label non-identifiability and (2) a broken forward recursion. Run it, read the diagnostics, fix each bug. Clean reference: `notebook.ipynb`; answer key: `BROKEN_BUGS.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import pytensor.tensor as pt
from pytensor import scan
import matplotlib.pyplot as plt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate(); y = data['y']; t = data['truth']
yt = pt.as_tensor_variable(y)

### BUG 1 — wrong forward recursion: max instead of log-sum-exp.

Replacing `logsumexp` with `max` computes the **Viterbi** best-path probability, not the marginal likelihood. Inference is then fit to the wrong objective and the parameters are biased.

In [ ]:
def forward_BUGGY(y, p01, p10, mu, sigma):
    yc = y[:, None]
    logem = -0.5*pt.log(2*np.pi*sigma**2) - 0.5*((yc-mu[None,:])/sigma)**2
    logP = pt.log(pt.stack([pt.stack([1-p01,p01]), pt.stack([p10,1-p10])]))
    a0 = pt.log(pt.as_tensor([0.5,0.5])) + logem[0]
    def step(logem_t, a_prev, logP):
        m = a_prev[:, None] + logP
        # BUG 1: should be pt.logsumexp(m, axis=0); max() drops the normalization
        return logem_t + pt.max(m, axis=0)
    seq, _ = scan(step, sequences=[logem[1:]], outputs_info=[a0],
                  non_sequences=[logP], return_updates=True)
    # BUG 1 (cont.): and the final reduction should be logsumexp, not max
    return pt.max(seq[-1])

### BUG 2 — unordered emission means: state-label non-identifiability.

In [ ]:
with pm.Model() as model:
    p01 = pm.Beta('p01', 2, 8); p10 = pm.Beta('p10', 2, 8)
    # BUG 2: no ordered transform -> labels can swap across chains
    mu = pm.Normal('mu', 0.0, 3.0, shape=2)
    sigma = pm.HalfNormal('sigma', 1.0)
    pm.Potential('hmm', forward_BUGGY(yt, p01, p10, mu, sigma))
    idata = pm.sample(draws=400, tune=400, chains=4, random_seed=RNG,
                      progressbar=False)

In [ ]:
# Expect: biased p01/p10 (wrong objective) AND high R-hat on mu (label swap).
print(az.summary(idata, var_names=['p01','p10','mu','sigma']))
print('true p01,p10 =', t['p01'], t['p10'])